In [ ]:
# Set Packing Notebook (alias-based, backend-agnostic test)

from optilab.aliases import Model, GRB, quicksum
import numpy as np

# -------------------------------------------------------------------------------
# Model
# -------------------------------------------------------------------------------
m = Model()
print("Backend:", m.backend_name)

# -------------------------------------------------------------------------------
# Problem data
# -------------------------------------------------------------------------------
# Reproducibility (important for solver comparisons)
rng = np.random.default_rng(42)

# Number of sets
n_sets = rng.integers(15, 40)

# Universe of elements
n_elements = rng.integers(20, 50)

# Random set-element incidence matrix
# membership = rng.integers(0, 2, size=(n_elements, n_sets))
membership = rng.choice([0, 1], size=(n_elements, n_sets), p=[0.9, 0.1])

# Ensure at least some structure (avoid empty model issues)
for j in range(n_sets):
    if membership[:, j].sum() == 0:
        membership[rng.integers(0, n_elements), j] = 1

# Profit for selecting each set
profit = rng.integers(1, 20, size=n_sets)

print("n sets:", n_sets)
print("n elements:", n_elements)
print("profit per set:", profit)

# -------------------------------------------------------------------------------
# Decision variables
# -------------------------------------------------------------------------------
x = [m.add_binary_var(name=f"x_{j}") for j in range(n_sets)]

# -------------------------------------------------------------------------------
# Constraint
# -------------------------------------------------------------------------------

# No element can be covered by more than one selected set
for i in range(n_elements):
    m.add_constraint(
        quicksum(membership[i][j] * x[j] for j in range(n_sets)) <= 1
    )

# -------------------------------------------------------------------------------
# Objective
# -------------------------------------------------------------------------------
m.set_objective(
    quicksum(profit[j] * x[j] for j in range(n_sets)),
    GRB.MAXIMIZE
)

# -------------------------------------------------------------------------------
# Solve
# -------------------------------------------------------------------------------
m.solve()

# -------------------------------------------------------------------------------
# Extract solution
# -------------------------------------------------------------------------------
solution = np.array([v.x for v in x])

print(f"Solution:       {solution}")
print("Total profit:", float(np.sum(profit * solution)))